<table>
  <tr>
    <td><div align="left"><font size="30">Machine Vision Toolbox for Python — Interactive Demo</font></div></td>
    <td><img src="https://github.com/petercorke/machinevision-toolbox-python/raw/main/docs/figs/VisionToolboxLogo_NoBackgnd@2x.png" width="400"></td>
  </tr>
</table>

A quick, runnable introduction to the toolbox. Works locally, on Google Colab, or entirely in your browser via [JupyterLite](https://jupyterlite.readthedocs.io) and [Pyodide](https://pyodide.org) -- no installation required either way. The cell below installs the toolbox the first time it is run (and will take a few seconds on Colab or in the browser).

In [ ]:
# MVTB_BOOTSTRAP_CELL -- sets up the environment (Colab / JupyterLite / local install); click to expand. Generated from docs/notebooks/_mvtb_nb_bootstrap.py by sync_bootstrap.py -- do not hand-edit.

"""Environment bootstrap for machinevision-toolbox-python's Jupyter notebooks.

Installs the toolbox (and reports how) across the three environments a notebook in
this folder might run in: a local Jupyter/VS Code install, Google Colab, and
JupyterLite (Pyodide/WASM, in-browser).

This file is the single source of truth for that logic. Every notebook's own
bootstrap cell is a generated copy of this file's content, produced by
sync_bootstrap.py -- see docs/notebooks/README.md for the full explanation.
"""

import subprocess
import sys
from pathlib import Path


async def ensure_installed() -> bool:
    """Install machinevision-toolbox-python if needed, and report the environment.

    :returns: True if running on Google Colab, False otherwise.
    """
    if sys.platform == "emscripten":
        import micropip

        await micropip.install(
            [
                "opencv-python",
                "spatialmath-python",
                "pgraph-python",
                "ansitable",
                "mvtb-data",
                "tqdm",
                "requests",
                "ipywidgets",
            ]
        )
        import cv2  # noqa: F401 - force cv2 into module registry before toolbox import

        wheels = sorted(Path("/pypi").glob("machinevision_toolbox_python-*.whl"))
        if wheels:
            # Prefer the wheel bundled with this JupyterLite site. Relative,
            # not absolute: a leading slash resolves against the origin, not
            # this site's own base URL (a GitHub Pages project subpath).
            await micropip.install(f"pypi/{wheels[-1].name}", deps=False)
        else:
            # Fall back to PyPI when running outside the published site layout.
            await micropip.install("machinevision-toolbox-python", deps=False)
        where, colab = "in browser", False
    else:
        try:
            import google.colab  # noqa: F401
        except ImportError:
            where, colab = "locally", False
        else:
            print("Installing machinevision-toolbox-python...")
            subprocess.run(
                [
                    sys.executable,
                    "-m",
                    "pip",
                    "install",
                    "-q",
                    "machinevision-toolbox-python",
                ],
                check=True,
            )
            where, colab = "on Colab", True

    import cv2

    import machinevisiontoolbox

    version = getattr(machinevisiontoolbox, "__version__", "unknown")
    opencv_version = getattr(cv2, "__version__", "unknown")
    print(f"Running {where} using MVTB v{version} with OpenCV {opencv_version}")
    return colab

COLAB = await ensure_installed()


## Images and pixels

The core class is `Image`.  Let's read one of the bundled images and inspect it.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

from machinevisiontoolbox import Image

mona = Image.Read("monalisa.png")
print(f"size: {mona.width} x {mona.height}, dtype: {mona.dtype}, colour: {mona.iscolor}")


Display the image inline with `disp()`, which uses matplotlib.

In [ ]:
mona.disp()

<div class="alert alert-info">
<strong>Note:</strong> In this environment it is not possible to examine pixel values by hovering the mouse cursor over an image.  This a limitation of Matplotlib in a Pyodide Web Worker.
</div>

## Smoothing

Apply a Gaussian blur and display original and result side-by-side.

In [ ]:
smooth = mona.smooth(sigma=3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
mona.disp(ax=axes[0], title="Original")
smooth.disp(ax=axes[1], title="Smoothed (sigma=3)")
plt.tight_layout()

## Greyscale and histograms

In [ ]:
grey = mona.mono()
print(f"grey: {grey.width} x {grey.height}, planes: {grey.nplanes}")

hist, x = grey.hist()
plt.figure()
plt.bar(x, hist, width=1, color="steelblue")
plt.xlabel("Pixel value")
plt.ylabel("Count")
plt.title("Greyscale histogram")
plt.tight_layout()

## Colour planes

Access individual colour planes by name — the toolbox tracks the colour order so
you never need to worry about BGR vs RGB.

In [ ]:
flowers = Image.Read("flowers1.png")
print(f"colour order: {flowers.colororder_str}")

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, plane in zip(axes, ["R", "G", "B"]):
    flowers.plane(plane).disp(ax=ax, title=plane, colormap="gray")
plt.tight_layout()

## Binary blobs

Load a binary image and find the blobs.

In [ ]:
sharks = Image.Read("shark2.png")
blobs = sharks.blobs()
print(blobs)

fig, ax = plt.subplots()
sharks.disp(ax=ax)
blobs.plot_box(ax, color="g")
blobs.plot_centroid(ax, "o", color="y")
plt.tight_layout()

## Camera model

Create a central perspective camera and project a 3-D point into the image plane.

In [ ]:
from machinevisiontoolbox import CentralCamera
from spatialmath import SE3

cam = CentralCamera(f=0.015, rho=10e-6, imagesize=[1280, 1024],
                    pp=[640, 512], name="mycamera")
print(cam)

P = [0.3, 0.4, 3.0]
p = cam.project_point(P)
print(f"Projected pixel: {p}")

# Shift camera 100 mm to the right
p2 = cam.project_point(P, pose=SE3(0.1, 0, 0))
print(f"Projected pixel (camera shifted): {p2}")